# Étape 2 — Présentation du problème

Dans ce notebook, nous présentons le problème choisi en suivant les cinq points : lien Kaggle, problématique, objectifs de prédiction, description des variables et défis.

## 1. Lien Kaggle et contexte

Nous avons retenu [House Prices – Advanced Regression Techniques](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/overview/description). La compétition porte sur des maisons situées à **Ames, dans l'Iowa, aux États-Unis**. Kaggle présente 79 caractéristiques explicatives pour prédire leur prix de vente. Les fichiers proposés pour ce projet sont présentés dans l'onglet [Data](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/data).

## 2. Problématique étudiée

Comment estimer le prix de vente d'une maison à partir des caractéristiques de son terrain, de sa localisation, de ses surfaces, de sa qualité, de son ancienneté et de ses équipements ? Nous souhaitons étudier les associations entre ces caractéristiques et les prix, puis vérifier dans quelle mesure elles permettent de prédire le prix d'autres maisons.

## 3. Objectifs de prédiction

L’objectif est d’**estimer le prix de vente d’une maison à partir de ses caractéristiques**. Il s’agit d’un problème de **régression supervisée**, puisque le modèle apprend à partir des données de `train.csv`, qui contiennent les caractéristiques des maisons ainsi que leur prix de vente connu, représenté par la variable cible numérique `SalePrice`. Le modèle utilisera ensuite les relations apprises pour prédire le prix des maisons de `test.csv`, dont les prix ne sont pas fournis.

Pour la phase de test, la variable `Id` sert uniquement à identifier chaque maison et à lui associer sa prédiction. Elle ne fait pas partie des **79 caractéristiques explicatives candidates**, parmi lesquelles aucune sélection définitive n’a encore été effectuée. Kaggle attend un fichier CSV contenant les colonnes `Id` et `SalePrice`, avec une prédiction en dollars américains pour chaque maison du test. Le fichier `sample_submission.csv` illustre le format attendu; ses valeurs ne correspondent pas aux véritables prix du test.

Les prédictions seront évaluées à l’aide de la **RMSE (*Root Mean Square Error*)**, ou racine de l’erreur quadratique moyenne, calculée **sur les logarithmes des prix** :

$$
\operatorname{RMSE}_{\log}
=
\sqrt{\frac{1}{n}\sum_{i=1}^{n}
\left(\log(\widehat{y}_i)-\log(y_i)\right)^2}.
$$

Dans cette formule, $\widehat{y}_i$ est le prix prédit et $y_i$ le prix réel. **Plus la RMSE est faible, plus les prédictions sont proches des prix réels.** Les écarts sont élevés au carré, ce qui donne davantage de poids aux grandes erreurs. L’utilisation du logarithme permet de tenir compte des écarts proportionnels : prédire le double du prix réel entraîne la même pénalité pour une maison abordable que pour une maison coûteuse.

## 4. Présentation des fichiers et aperçu des données

| Fichier | Rôle |
|---|---|
| `train.csv` | Données d’apprentissage contenant les caractéristiques des maisons et leur prix de vente (`SalePrice`), la valeur à prédire. |
| `test.csv` | Données utilisées pour l’évaluation sur Kaggle. Elles contiennent les caractéristiques des maisons, mais pas leur prix de vente. Nous devons prédire ces prix et les soumettre à Kaggle. |
| `data_description.txt` | Description des caractéristiques et de la signification des codes utilisés dans les données. |
| `sample_submission.csv` | Exemple du fichier à soumettre à Kaggle, avec les colonnes `Id` et `SalePrice`. |

### Lecture des données

Nous lisons d’abord toutes les valeurs des CSV **sous forme de texte**, avec les paramètres `dtype=str` et `keep_default_na=False`. Cela permet de conserver les données telles qu’elles apparaissent dans les fichiers, notamment les codes `NA`, `None` et les cellules vides.

Ces codes doivent être interprétés selon la description de chaque caractéristique. Par exemple, `None` dans `MasVnrType` indique qu’une maison n’a pas de parement de maçonnerie. De même, `NA` peut indiquer l’absence d’un équipement plutôt qu’une information manquante.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

# Repérer la racine depuis le dossier courant ou l'un de ses parents.
current = Path.cwd().resolve()
root = next(
    (path for path in (current, *current.parents) if (path / "pyproject.toml").is_file()),
    None,
)
if root is None:
    raise FileNotFoundError("Ouvrir ce notebook depuis le dépôt MAP6014.")

data_dir = root / "data"
read_options = {"dtype": str, "keep_default_na": False}
train = pd.read_csv(data_dir / "train.csv", **read_options)
test = pd.read_csv(data_dir / "test.csv", **read_options)
sample_submission = pd.read_csv(data_dir / "sample_submission.csv", **read_options)
description_text = (data_dir / "data_description.txt").read_text(encoding="utf-8")

assert train.shape == (1460, 81)
assert test.shape == (1459, 80)
assert "SalePrice" not in test.columns
assert list(test.columns) == [name for name in train.columns if name != "SalePrice"]
assert list(sample_submission.columns) == ["Id", "SalePrice"]
assert sample_submission["Id"].tolist() == test["Id"].tolist()
assert train["Id"].is_unique and test["Id"].is_unique
assert len(train.columns.difference(["Id", "SalePrice"])) == 79

dimensions = pd.DataFrame(
    [("train.csv", *train.shape), ("test.csv", *test.shape),
     ("sample_submission.csv", *sample_submission.shape)],
    columns=["Fichier", "Lignes", "Colonnes"],
)
display(dimensions)

Le fichier `train.csv` contient **1 460 maisons et 81 colonnes**, contre **1 459 maisons et 80 colonnes** pour `test.csv`. La différence vient de la colonne `SalePrice`, présente uniquement dans les données d’apprentissage. Le fichier `data_description.txt` contient du texte explicatif plutôt qu’un tableau de données.

### Les cinq premières observations

Le tableau ci-dessous présente les **cinq premières maisons de `train.csv`**. Vous pouvez le faire défiler horizontalement pour consulter les 81 colonnes. Cet aperçu permet de voir comment les données sont organisées.

In [ ]:
preview = train.head(5)
assert preview.shape == (5, 81)

preview_html = preview.to_html(index=False, max_rows=None, max_cols=None)
display(HTML(
    '<div role="region" aria-label="Cinq maisons, toutes les variables" '
    'tabindex="0" style="width:100%;max-width:100%;overflow-x:auto;'
    'border:1px solid #ddd;padding:8px;">'
    + preview_html
    + '</div>'
))

### Vérification des codes `NA`, `None` et des cellules vides

Le tableau ci-dessous compte les valeurs `NA`, `None` et les cellules vides dans quelques colonnes de `train.csv` et de `test.csv`. Cette vérification permet de s’assurer que ces codes ont été conservés lors de la lecture des fichiers.

In [ ]:
raw_checks = []
for file_name, frame in [("train", train), ("test", test)]:
    for column in ["MasVnrType", "Alley", "MSZoning", "LotFrontage", "GarageYrBlt"]:
        raw_checks.append({
            "Fichier": file_name,
            "Variable": column,
            "Code NA": int(frame[column].eq("NA").sum()),
            "Code None": int(frame[column].eq("None").sum()),
            "Cellule vide": int(frame[column].eq("").sum()),
        })
display(pd.DataFrame(raw_checks))
assert train["MasVnrType"].eq("None").any()
assert train["Alley"].eq("NA").any()

## 5. Description des variables

Les données d’apprentissage contiennent **81 colonnes** : l’identifiant de la maison (`Id`), **79 caractéristiques** et le prix de vente à prédire (`SalePrice`). Nous regroupons ces colonnes par thème, avec une courte description en français et leur unité, lorsqu’elle s’applique. Le choix des caractéristiques utilisées pour la prédiction sera fait plus tard.

Les descriptions proviennent principalement de [data_description.txt](../data/data_description.txt). Nous conservons les noms des colonnes présents dans les CSV. Le fichier source fournit le détail des codes et des catégories.

### Identifiant et prix à prédire

- `Id` : numéro permettant d’identifier chaque maison.
- `SalePrice` : prix de vente en dollars américains (USD), fourni uniquement dans `train.csv`.

### Terrain et localisation

- `MSSubClass` : Type de logement vendu, codé par des nombres qui ne sont pas des quantités (sans unité).
- `MSZoning` : Classification du zonage (sans unité).
- `LotFrontage` : Longueur de façade du terrain donnant sur la rue — unité : pieds (ft).
- `LotArea` : Superficie du terrain — unité : pieds carrés (ft²).
- `Street` : Type de voie donnant accès au terrain (sans unité).
- `Alley` : Type de ruelle donnant accès au terrain (sans unité).
- `LotShape` : Régularité de la forme du terrain (sans unité).
- `LandContour` : Relief et position du terrain (sans unité).
- `Utilities` : Services publics disponibles (sans unité).
- `LotConfig` : Configuration du lot par rapport aux rues et aux autres lots (sans unité).
- `LandSlope` : Importance de la pente du terrain (sans unité).
- `Neighborhood` : Quartier à l'intérieur des limites d'Ames (sans unité).
- `Condition1` : Première condition de proximité, par exemple route ou voie ferrée (sans unité).
- `Condition2` : Deuxième condition de proximité lorsqu'il y en a plusieurs (sans unité).

### Bâtiment et ancienneté

- `BldgType` : Type de bâtiment résidentiel (sans unité).
- `HouseStyle` : Style et organisation des niveaux du logement (sans unité).
- `OverallQual` : Évaluation globale des matériaux et des finitions, de 1 à 10 — unité : Score de 1 à 10.
- `OverallCond` : Évaluation globale de l'état de la maison, de 1 à 10 — unité : Score de 1 à 10.
- `YearBuilt` : Année de construction initiale — unité : année.
- `YearRemodAdd` : Année de rénovation ou d'agrandissement; année de construction s'il n'y en a pas — unité : année.

### Enveloppe et fondations

- `RoofStyle` : Forme du toit (sans unité).
- `RoofMatl` : Matériau de couverture du toit (sans unité).
- `Exterior1st` : Revêtement extérieur principal (sans unité).
- `Exterior2nd` : Autre revêtement extérieur si plusieurs matériaux sont présents (sans unité).
- `MasVnrType` : Type de parement de maçonnerie (sans unité).
- `MasVnrArea` : Superficie du parement de maçonnerie — unité : pieds carrés (ft²).
- `ExterQual` : Qualité des matériaux extérieurs (sans unité).
- `ExterCond` : État actuel des matériaux extérieurs (sans unité).
- `Foundation` : Type de fondation (sans unité).

### Sous-sol

- `BsmtQual` : Évaluation de la hauteur du sous-sol — unité : Classes de hauteur en pouces.
- `BsmtCond` : État général du sous-sol (sans unité).
- `BsmtExposure` : Exposition des murs du sous-sol vers l'extérieur, notamment au niveau du jardin (sans unité).
- `BsmtFinType1` : Évaluation du premier type d'espace aménagé au sous-sol (sans unité).
- `BsmtFinSF1` : Superficie aménagée de type 1 au sous-sol — unité : pieds carrés (ft²).
- `BsmtFinType2` : Évaluation du second type d'espace aménagé au sous-sol (sans unité).
- `BsmtFinSF2` : Superficie aménagée de type 2 au sous-sol — unité : pieds carrés (ft²).
- `BsmtUnfSF` : Superficie non aménagée du sous-sol — unité : pieds carrés (ft²).
- `TotalBsmtSF` : Superficie totale du sous-sol — unité : pieds carrés (ft²).

### Chauffage et électricité

- `Heating` : Type de chauffage (sans unité).
- `HeatingQC` : Qualité et état du chauffage (sans unité).
- `CentralAir` : Présence d'une climatisation centrale (sans unité).
- `Electrical` : Type d'installation électrique (sans unité).

### Surfaces et pièces

- `1stFlrSF` : Superficie du premier niveau, correspondant au rez-de-chaussée — unité : pieds carrés (ft²).
- `2ndFlrSF` : Superficie du deuxième niveau — unité : pieds carrés (ft²).
- `LowQualFinSF` : Superficie aménagée de faible qualité, tous niveaux confondus — unité : pieds carrés (ft²).
- `GrLivArea` : Superficie habitable au-dessus du niveau du sol — unité : pieds carrés (ft²).
- `BsmtFullBath` : Nombre de salles de bain complètes au sous-sol — unité : nombre.
- `BsmtHalfBath` : Nombre de salles d'eau au sous-sol — unité : nombre.
- `FullBath` : Nombre de salles de bain complètes hors sous-sol — unité : nombre.
- `HalfBath` : Nombre de salles d'eau hors sous-sol — unité : nombre.
- `BedroomAbvGr` : Nombre de chambres hors sous-sol — unité : nombre.
- `KitchenAbvGr` : Nombre de cuisines hors sous-sol — unité : nombre.
- `KitchenQual` : Qualité de la cuisine (sans unité).
- `TotRmsAbvGrd` : Nombre total de pièces hors sous-sol, sans les salles de bain — unité : nombre.
- `Functional` : Niveau de fonctionnalité du logement, selon les défauts constatés (sans unité).
- `Fireplaces` : Nombre de foyers — unité : nombre.
- `FireplaceQu` : Qualité du foyer (sans unité).

### Garage et accès

- `GarageType` : Emplacement ou type de garage (sans unité).
- `GarageYrBlt` : Année de construction du garage — unité : année.
- `GarageFinish` : Degré de finition intérieure du garage (sans unité).
- `GarageCars` : Capacité du garage en nombre de voitures — unité : voitures.
- `GarageArea` : Superficie du garage — unité : pieds carrés (ft²).
- `GarageQual` : Qualité du garage (sans unité).
- `GarageCond` : État du garage (sans unité).
- `PavedDrive` : Degré de pavage de l'entrée (sans unité).

### Aménagements extérieurs

- `WoodDeckSF` : Superficie de la terrasse en bois — unité : pieds carrés (ft²).
- `OpenPorchSF` : Superficie de la galerie ouverte — unité : pieds carrés (ft²).
- `EnclosedPorch` : Superficie de la galerie fermée — unité : pieds carrés (ft²).
- `3SsnPorch` : Superficie de la galerie trois saisons — unité : pieds carrés (ft²).
- `ScreenPorch` : Superficie de la galerie avec moustiquaires — unité : pieds carrés (ft²).
- `PoolArea` : Superficie de la piscine — unité : pieds carrés (ft²).
- `PoolQC` : Qualité de la piscine (sans unité).
- `Fence` : Type de clôture et niveau de protection de l'intimité (sans unité).
- `MiscFeature` : Équipement particulier non décrit dans les autres catégories (sans unité).
- `MiscVal` : Valeur de l'équipement particulier — unité : dollars ($ dans le dictionnaire).

### Vente

- `MoSold` : Mois de la vente — unité : mois (MM).
- `YrSold` : Année de la vente — unité : année (YYYY).
- `SaleType` : Type de transaction (sans unité).
- `SaleCondition` : Conditions particulières de la vente (sans unité).

## 6. Défis associés au problème

Plusieurs éléments devront être examinés pour préparer les données et construire un modèle fiable. Le tableau suivant présente les principaux défis et les vérifications prévues. Certains problèmes, comme les catégories rares ou les valeurs inhabituelles, restent à confirmer.

| Défi | Pourquoi est-ce important? | Vérification prévue |
|---|---|---|
| Différents types de variables | Les données contiennent des mesures, des années et des catégories. Un code numérique ne représente pas toujours une quantité. | Déterminer comment traiter chaque variable selon sa signification. |
| Absence d’équipement ou information manquante | Un code `NA` ou `None` peut indiquer qu’un équipement est absent plutôt qu’une information inconnue. | Vérifier la signification de ces codes pour chaque variable. |
| Catégories peu représentées | Certaines catégories pourraient être rares ou présentes dans un seul des deux fichiers. | Compter les maisons par catégorie et comparer `train.csv` à `test.csv`. |
| Unités et échelles différentes | Les surfaces, les années et les nombres de pièces ont des valeurs très différentes. | Vérifier si le modèle choisi nécessite de ramener les valeurs à une échelle comparable. |
| Répartition des prix | Quelques maisons très chères pourraient influencer les erreurs du modèle. | Observer la répartition des prix et de leurs logarithmes. |
| Valeurs inhabituelles et caractéristiques liées | Certaines maisons pourraient avoir des valeurs extrêmes, et plusieurs caractéristiques pourraient fournir une information similaire. | Examiner ces valeurs et les relations entre les caractéristiques avant de décider de leur traitement. |
| Surapprentissage | Le modèle pourrait bien prédire les maisons utilisées pour apprendre, mais moins bien les nouvelles maisons. | Réserver une partie des données pour évaluer les prédictions sur des maisons non utilisées pendant l’apprentissage. |
| Application à d’autres marchés | Les données concernent Ames. Les prix peuvent dépendre d’autres facteurs ailleurs ou à une autre période. | Préciser les limites du modèle : ces données ne suffisent pas à vérifier sa fiabilité dans notre région. |

Pour garder une évaluation fiable, **les données de validation ne serviront pas à apprendre les règles de préparation des données**, comme les valeurs utilisées pour remplacer les informations manquantes.

## Références

- [Consignes originales MAP6014](../docs/travail-de-mi-session-v2.pdf), page 2, étape 2. Le document porte le titre « travail de mi-session »; nous l'utilisons comme consigne du présent travail de fin de session.
- [Kaggle — Présentation demandée](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/overview/description) et [page principale consultable](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques), rubriques « Competition Description », « Goal », « Metric » et « Submission File Format ».
- [Kaggle — Données](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/data), sous-page sans texte exploitable lors de la consultation.
- [Description locale des variables](../data/data_description.txt), source des descriptions et modalités anglaises. Les fichiers CSV locaux `train.csv`, `test.csv` et `sample_submission.csv` servent aux vérifications et affichages. Ils restent locaux et ne sont pas versionnés.
- [Étape 1 — Choix du problème](01_choix_probleme.ipynb), pour notre motivation personnelle.

**Convention de dépôt.** Les sorties de calcul sont effacées avant le commit, conformément au [guide de collaboration](../docs/collaboration.md). Exécuter les cellules dans l'ordre avec le kernel `.venv` pour afficher les dimensions, l'aperçu horizontal et les contrôles. Les vérifications de préparation ont utilisé pandas dans l'environnement fourni avec Codex; l'exécution dans un kernel Jupyter n'y était pas disponible.